# Kobe Lip-Sync — JoyVASA on Free Colab (T4 GPU)

**Why this one:** JoyVASA is the only free open-source model explicitly trained to animate **animal** faces
(its paper: *"extends beyond human portraits to animate animal faces seamlessly"*), and it has a built-in
`animation_mode: animal` flag. It uses LivePortrait internally as the renderer + accepts audio directly.

**Cost: $0.** Colab free tier T4. Produces an 8-10s talking clip of Kobe.

**How to run:** Runtime → Change runtime type → **T4 GPU** → then Runtime → Run all.

Verified against the real repo on 2026-09-05 — the actual CLI is `--reference` / `--audio` / `--animation_mode animal`
(NOT the `--source_image/--driven_audio` flags some AI guides invent).

## 1. Clone + install (3-5 min)

In [ ]:
!git clone https://github.com/jdh-algo/JoyVASA.git
%cd JoyVASA
!apt-get install -y ffmpeg git-lfs
!git lfs install
!pip install -q tyro==0.8.5 accelerate==0.28.0 bitsandbytes==0.43.1 diffusers==0.27.2 \
    einops==0.8.0 librosa==0.10.2.post1 mediapipe==0.10.14 imageio-ffmpeg pykalman \
    opencv-python scipy scikit-image onnxruntime-gpu
print('✓ deps installed')

## 2. Model checkpoints (~10 GB, one time per session)

In [ ]:
import os
os.makedirs('pretrained_weights', exist_ok=True)

# JoyVASA checkpoints
!git clone https://huggingface.co/jdh-algo/JoyVASA pretrained_weights/JoyVASA
!git clone https://huggingface.co/facebook/wav2vec2-base-960h pretrained_weights/wav2vec2-base-960h

# LivePortrait renderer weights (same ones we already use locally)
!git clone https://huggingface.co/KwaiVGI/LivePortrait pretrained_weights/liveportrait

print('✓ checkpoints ready')

## 3. Upload Kobe + audio

Drag two files into the Colab **Files** panel (left sidebar), into `JoyVASA/`:
- `kobe.png` — clear, front-facing, mouth relaxed (use `Brands/Kobe/refs/kobe-viewsai.png`)
- `kobe.wav` — your 8-10s line (Kokoro TTS output converted to wav)

Then run the cell below.

In [ ]:
from google.colab import files
print('upload kobe.png and kobe.wav')
files.upload()

## 4. Run inference — ANIMAL mode

In [ ]:
# animation_mode=animal is the key flag (plus stitching off, as recommended for animals)
!python inference.py \
  --reference kobe.png \
  --audio kobe.wav \
  --animation_mode animal \
  --output_dir outputs \
  --flag_stitching False

import glob, shutil
vids = sorted(glob.glob('outputs/**/*.mp4', recursive=True), key=os.path.getmtime)
print('produced:', vids[-1] if vids else 'NONE')
if vids:
    shutil.copy(vids[-1], 'kobe_talking.mp4')
    files.download('kobe_talking.mp4')

## Automate the daily run

- Save this notebook to your Drive: **File → Save a copy in Drive** → then each day: open → Runtime → Run all (~6-8 min incl. setup).
- Skip the checkpoint cell after the first run in the same session (files persist during a session).
- **Even better:** use Colab's "Run all" with the *same* session — installs only happen once per session.

### When the clip lands on your Mac
```bash
# drop it into the movement library, then:
python3 ~/ViewsOSComplete/scripts/free-stack-videos/assemble_kobe.py \
  --movement <new-clip> --line "..." --title "..." --cta "Comment AI"
```
That gives you Kokoro voice + safe-zone text + 12 platform renders.